# Google Search Collection (Batch Mode)

Run 10 queries at a time to avoid CAPTCHAs. Each batch appends to the same CSV.

In [1]:
!pip install undetected-chromedriver pandas tldextract selenium -q

In [1]:
import csv
import time
import random
import subprocess
import platform
import re
import urllib.parse
from datetime import datetime, timezone
from pathlib import Path

import tldextract
import pandas as pd

print("Imports OK ✓")

Imports OK ✓


## Step 1: Detect Chrome Version

In [2]:
def get_chrome_version():
    system = platform.system()
    try:
        if system == "Darwin":
            result = subprocess.run(
                ["/Applications/Google Chrome.app/Contents/MacOS/Google Chrome", "--version"],
                capture_output=True, text=True)
        elif system == "Windows":
            result = subprocess.run(
                ["reg", "query", r"HKEY_CURRENT_USER\Software\Google\Chrome\BLBeacon", "/v", "version"],
                capture_output=True, text=True)
        else:
            result = subprocess.run(["google-chrome", "--version"], capture_output=True, text=True)
        match = re.search(r"(\d+)\.\d+\.\d+\.\d+", result.stdout)
        if match:
            major = int(match.group(1))
            print(f"Chrome detected: {result.stdout.strip()} (major: {major})")
            return major
    except Exception as e:
        print(f"Auto-detect failed: {e}")
    print("Set CHROME_VERSION manually from chrome://version")
    return None

CHROME_VERSION = get_chrome_version()

Chrome detected: Google Chrome 146.0.7680.178 (major: 146)


## Step 2: Configuration

In [3]:
# If auto-detect failed, set manually:
# CHROME_VERSION = 136

INPUT_CSV    = "participant_queries.csv"
OUTPUT_DIR   = "google_results"
MAX_RESULTS  = 10
BASE_DELAY   = 5.0
CAPTCHA_WAIT = 120

TASKS = {
    "chicken":       "NPC",
    "fetus":         "ALowC",
    "states":        "AHighC",
    "breastcancer":  "MisC",
    # "washchicken":   "NPO",
    # "doctor":        "ALowO",
    # "bishop":        "AHighO",
    # "plannedparent": "MisO",
}

Path(OUTPUT_DIR).mkdir(exist_ok=True)

df = pd.read_csv(INPUT_CSV, dtype=str)
print(f"Loaded {len(df)} participants")
for task, col in TASKS.items():
    if col in df.columns:
        valid = df[col].dropna().apply(lambda x: len(str(x).strip()) > 1).sum()
        print(f"  {task:15s}: {valid}/227 valid")

Loaded 227 participants
  chicken        : 223/227 valid
  fetus          : 223/227 valid
  states         : 223/227 valid
  breastcancer   : 223/227 valid


## Step 3: Browser Setup

In [4]:
def make_driver():
    # Attempt 1: undetected-chromedriver
    try:
        import undetected_chromedriver as uc
        options = uc.ChromeOptions()
        options.add_argument("--no-first-run")
        options.add_argument("--no-default-browser-check")
        options.add_argument("--disable-extensions")
        options.add_argument("--window-size=1280,900")
        driver = uc.Chrome(options=options, version_main=CHROME_VERSION, headless=False)
        print("  Browser: undetected-chromedriver ✓")
        return driver
    except Exception as e:
        print(f"  UC failed: {e}")
        print("  Falling back to Selenium + stealth...")

    # Attempt 2: regular Selenium
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    options = Options()
    options.add_argument("--no-first-run")
    options.add_argument("--no-default-browser-check")
    options.add_argument("--disable-extensions")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument("--window-size=1280,900")
    options.add_argument(
        "--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36")
    driver = webdriver.Chrome(options=options)
    stealth_js = '''
        Object.defineProperty(navigator, 'webdriver', {get: () => undefined});
        window.chrome = { runtime: {} };
        Object.defineProperty(navigator, 'plugins', {get: () => [1, 2, 3, 4, 5]});
        Object.defineProperty(navigator, 'languages', {get: () => ['en-US', 'en']});
    '''
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {"source": stealth_js})
    print("  Browser: Selenium + stealth ✓")
    return driver

# Quick test
print("Testing browser...")
d = make_driver()
d.get("https://www.google.com")
time.sleep(2)
print(f"  Title: {d.title}")
d.quit()
print("  Test passed ✓")

Testing browser...
  UC failed: Message: Can not connect to the Service /Users/ushnamalik/Library/Application Support/undetected_chromedriver/undetected_chromedriver

  Falling back to Selenium + stealth...


KeyboardInterrupt: 

## Step 4: Helpers & Scraper

In [6]:
def build_url(query):
    params = {"q": query, "hl": "en", "gl": "us", "num": "10", "pws": "0", "udm": "14"}
    return "https://www.google.com/search?" + urllib.parse.urlencode(params)

def get_domain(url):
    try: return urllib.parse.urlparse(url).netloc.replace("www.", "").strip()
    except: return ""

def get_reg_domain(url):
    try:
        ext = tldextract.extract(url)
        return f"{ext.domain}.{ext.suffix}" if ext.suffix else ext.domain
    except: return ""

CAPTCHA_WORDS = ["unusual traffic", "captcha", "recaptcha",
                 "are you a robot", "not a robot", "automated queries"]

def handle_captcha(driver):
    page = driver.page_source.lower()
    if not any(w in page for w in CAPTCHA_WORDS):
        return False
    print("\n    ⚠️  CAPTCHA — solve it in the browser window")
    for sec in range(CAPTCHA_WAIT):
        time.sleep(1)
        try:
            if not any(w in driver.page_source.lower() for w in CAPTCHA_WORDS):
                print(f"    ✓ Solved after {sec+1}s")
                time.sleep(2)
                return True
        except: pass
        if sec % 15 == 14:
            print(f"    Waiting... ({sec+1}s)")
    print("    ✗ Timeout")
    return True

JS_EXTRACT = """
const results = [];
const seen = new Set();
const MAX = arguments[0];
const links = document.querySelectorAll('a:has(h3)');
for (const a of links) {
    if (results.length >= MAX) break;
    const href = a.href || '';
    if (!href.startsWith('http') || href.includes('google.com/search')
        || href.includes('accounts.google') || href.includes('support.google')
        || href.includes('maps.google') || seen.has(href)) continue;
    seen.add(href);
    const h3 = a.querySelector('h3');
    const title = h3 ? h3.innerText.trim() : '';
    if (!title) continue;
    let snippet = '';
    let box = a.closest('div[data-sokoban], div[data-hveid], div.g, div.tF2Cxc, div.MjjYud');
    if (!box) box = a.parentElement?.parentElement?.parentElement?.parentElement;
    if (box) {
        for (const sel of ['div[data-sncf] span','div.VwiC3b','div.VwiC3b span',
            'span.aCOpRe','div.IsZvec','div[style*="-webkit-line-clamp"]',
            'div.ITZIwc','span.hgKElc']) {
            const el = box.querySelector(sel);
            if (el && el.innerText.trim().length > 20) { snippet = el.innerText.trim(); break; }
        }
        if (!snippet) {
            for (const d of box.querySelectorAll('div, span')) {
                const t = d.innerText?.trim() || '';
                if (t.length > 40 && t !== title && !t.includes(title) && !t.startsWith('http'))
                    { snippet = t; break; }
            }
        }
    }
    results.push({title, url: href, snippet});
}
return results;
"""

def scrape(driver):
    try:
        return driver.execute_script(JS_EXTRACT, MAX_RESULTS)
    except Exception as e:
        print(f"    JS error: {e}")
        return []

print("Helpers & scraper defined ✓")

Helpers & scraper defined ✓


## Step 5: Batch Collection Function

Collects a slice of queries and **appends** to the output CSV.

In [7]:
def collect_batch(task_name, col_name, df, start_at, batch_size=10):
    """Collect a batch of queries starting at start_at."""
    
    output_csv = f"{OUTPUT_DIR}/google_{task_name}_results.csv"
    end_at = min(start_at + batch_size, len(df))
    
    queries = [str(v).strip() if pd.notna(v) else "" for v in df[col_name]]
    batch_queries = queries[start_at:end_at]

    print(f"\n{'='*55}")
    print(f"  {task_name} — queries {start_at+1} to {end_at} of {len(queries)}")
    print(f"  → {output_csv}")
    print(f"{'='*55}")

    fields = ["query_index","col","query","rank","title","url",
              "domain","registrable_domain","snippet","n_results","timestamp"]

    driver = make_driver()
    driver.delete_all_cookies()
    captcha_count = 0

    try:
        # Append mode: write header only if file doesn't exist yet
        write_header = not Path(output_csv).exists()
        with open(output_csv, "a", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=fields)
            if write_header:
                w.writeheader()

            for i, query in enumerate(batch_queries):
                qi = start_at + i + 1   # global query index (1-based)

                if len(query) <= 1:
                    print(f"  [{qi:>3}] SKIPPED")
                    w.writerow({"query_index":qi,"col":col_name,"query":query,
                        "rank":None,"title":"SKIPPED","url":"","domain":"",
                        "registrable_domain":"","snippet":"","n_results":0,
                        "timestamp":datetime.now(timezone.utc).isoformat()})
                    f.flush()
                    continue

                print(f"  [{qi:>3}] {query[:55]!r}", end="")

                try:
                    driver.get(build_url(query))
                    time.sleep(random.uniform(2.0, 3.5))

                    if handle_captcha(driver):
                        captcha_count += 1
                        if captcha_count >= 3:
                            print("\n    Cooling down 30s...")
                            time.sleep(30)
                            captcha_count = 0

                    raw = scrape(driver)
                    ts = datetime.now(timezone.utc).isoformat()

                    if raw:
                        for rank, r in enumerate(raw, 1):
                            w.writerow({
                                "query_index":qi, "col":col_name, "query":query,
                                "rank":rank, "title":r["title"], "url":r["url"],
                                "domain":get_domain(r["url"]),
                                "registrable_domain":get_reg_domain(r["url"]),
                                "snippet":r["snippet"],
                                "n_results":len(raw), "timestamp":ts})
                        print(f" → {len(raw)} results")
                    else:
                        w.writerow({"query_index":qi,"col":col_name,"query":query,
                            "rank":None,"title":"","url":"","domain":"",
                            "registrable_domain":"","snippet":"",
                            "n_results":0,"timestamp":ts})
                        print(" → 0")
                    f.flush()

                except Exception as e:
                    print(f" ERROR: {e}")
                    w.writerow({"query_index":qi,"col":col_name,"query":query,
                        "rank":None,"title":f"ERROR: {e}","url":"","domain":"",
                        "registrable_domain":"","snippet":"","n_results":0,
                        "timestamp":datetime.now(timezone.utc).isoformat()})
                    f.flush()

                if i < len(batch_queries) - 1:
                    time.sleep(max(3.0, BASE_DELAY + random.uniform(-1.0, 3.0)))

    finally:
        driver.quit()

    print(f"\n  ✓ Batch done. Next run: START_AT = {end_at}")
    return end_at

print("Batch collection function defined ✓")

Batch collection function defined ✓


## Step 6: Run a Batch

**Re-run this cell** for each batch. After each run, bump `START_AT` to the number printed.

When `START_AT` reaches 227, the task is done — change `TASK_NAME` and reset to 0.

In [ ]:
# ┌──────────────────────────────────────────────┐
# │  CHANGE THESE TWO VALUES BETWEEN RUNS        │
# └──────────────────────────────────────────────┘
TASK_NAME  = "states"    # chicken, fetus, states, breastcancer
START_AT   = 46       # bump by 10 each run: 0, 10, 20, ...
BATCH_SIZE = 10
# ────────────────────────────────────────────────

col = TASKS[TASK_NAME]
next_start = collect_batch(TASK_NAME, col, df, START_AT, BATCH_SIZE)
print(f"\n  ➡️  Set START_AT = {next_start} and re-run this cell")


  states — queries 47 to 56 of 227
  → google_results/google_states_results.csv


## Check Progress

In [ ]:
for task in TASKS:
    path = f"{OUTPUT_DIR}/google_{task}_results.csv"
    try:
        r = pd.read_csv(path)
        done = r["query_index"].nunique()
        ok   = r[r["n_results"] > 0]["query_index"].nunique()
        print(f"  {task:15s}: {done}/227 collected, {ok} with results")
    except FileNotFoundError:
        print(f"  {task:15s}: not started")

## Reset a Task (if needed)
Delete the CSV to start over.

In [ ]:
# Uncomment to reset a specific task:
# Path(f"{OUTPUT_DIR}/google_chicken_results.csv").unlink(missing_ok=True)
# print("Reset done")